# Training and Testing Dataset

In [ ]:
import pandas as pd
from jupyter_core.migrate import regex
from pandas import DataFrame
from sympy.strategies.core import switch

datasets = [['duplicate', 'train', pd.read_csv('../data/duplicate_detect_train.csv')],
            ['duplicate', 'test', pd.read_csv('../data/duplicate_detect_test.csv')],
            ['unique', 'train', pd.read_csv('../data/unique_detect_train.csv')],
            ['unique', 'test', pd.read_csv('../data/unique_detect_test.csv')]]
for index, [distribution, dataset_name, df] in enumerate(datasets):
    df_size = len(df)
    no_size = len(df[df['label'] == 'no'])
    yes_size = len(df[df['label'] == 'yes'])
    project_size = len(df['repository'].unique())

    other_df = datasets[index + 1 if index % 2 == 0 else index - 1][-1]
    other_df_size = len(other_df)
    other_no_size = len(other_df[other_df['label'] == 'no'])
    other_yes_size = len(other_df[other_df['label'] == 'yes'])

    overall_project_size = len(set(df['repository'].unique()).union(set(other_df['repository'].unique())))
    print(
        f"{distribution} & {dataset_name} & {df_size:,} ({df_size / (df_size + other_df_size) * 100:.1f}) & {no_size:,} ({no_size / (no_size + other_no_size) * 100: .1f}) & {yes_size:,} ({yes_size / (yes_size + other_yes_size) * 100:.1f}) & {yes_size / df_size * 100:.1f} & {project_size} ({project_size / overall_project_size * 100:.1f}) \\\\")



# non-LLM Result

In [ ]:
import pandas as pd
from pandas import DataFrame
def bold_max(column_name: str, row_df: DataFrame, df : DataFrame, decimal_digit: int = 1):
    value = row_df[column_name]
    if value == df[column_name].max():
        return "\\textbf{" + f"{value: .1f}" + "}"
    else:
        return value

df = pd.read_csv('../cache/output/detect_output_metrics-8.csv')
model_map = {'pretrained-potdar-Pattern': 'Pattern',
             'pretrained-TM': 'TM',
             'trained-TM': 'TM',
             'pretrained-NLP': 'NLP',
             'trained-NLP': 'NLP',
             'pretrained-MAT': 'MAT',
             # 'trained-MAT': 'MAT'
    # ,
    #          'pretrained-liu-detector': 'Liu',
    #          'trained-liu-detector': 'Liu'
             }

MODEL_ORDER = ['Pattern',  'NLP', 'TM', 'MAT']

full_df = df[(df['model'].isin(model_map.keys())) & (df['metric'] == 'yes')].copy()
full_df['model_short'] = full_df['model'].apply(lambda x: model_map[x])
full_df['model_order'] = full_df['model_short'].apply(lambda x: MODEL_ORDER.index(x))
full_df['retrained'] = full_df['model'].apply(lambda x: 'no' if 'pretrained' in x else 'yes')
full_df.loc[full_df['model_short'].isin({'Pattern', 'MAT'}), "retrained"] = '-'
full_df['dataset'] = full_df['dataset'].map({'unique': 'Deduplicate', 'duplicate': 'Original'})

for group_index, (group_name, df) in enumerate( sorted(full_df.groupby('dataset'), key=lambda x: x[0], reverse=True)):
    # print("\\hline") if group_index > 0 else None
    print("\\hline")
    df = df.sort_values(by=['model_order', 'retrained'], ascending=[True, True])

    for loop_index, (index, row) in enumerate(df.iterrows()):
        model_url = row['model']
        # print(f"{len(df)} : {index}")
        is_middle_index = len(df) // 2 == loop_index
        print(
            f"{row['dataset'] if is_middle_index else ''}  & {row['model_short']} & {row['retrained']}    &   {bold_max('precision', row, df)} & {bold_max('recall', row, df)} & {bold_max('f1-score', row, df)}   & {row['support'] if is_middle_index else ''} \\\\")
print("\\hline")


# OSS Model

In [10]:
import pandas as pd
from pandas import DataFrame
def bold_max(column_name: str, row_df: DataFrame, df : DataFrame, decimal_digit: int = 1):
    value = row_df[column_name]
    if value == df[column_name].max():
        return "\\textbf{" + f"{value: .1f}" + "}"
    else:
        return value

def convert_prompt(prompt):
        if prompt ==  'definition':
            return 'No Keyword'
        elif prompt == 'mat':
            return 'MAT'
        elif prompt == 'jitterbug':
            return 'Jitterbug'
        elif prompt == 'gpt':
            return 'GPT 4'
        else:
            return 'Ours'


MODEL_ORDER = ['t5-xl', 't5-xxl']
df = pd.read_csv('../cache/output/detect_output_metrics-10.csv')
full_df = df[(df['model'].str.contains('|'.join(['t5-xl', 't5-xxl']), regex= True)) & (df['metric'] == 'yes')].copy()
full_df['model_short'] = full_df['model'].apply(lambda x: x[x.index('t5-') : x.index('-', x.index('t5-') + 4)])
full_df['model_order'] = full_df['model_short'].apply(lambda x: MODEL_ORDER.index(x))
full_df['shot'] = full_df['model'].apply(lambda x: int(x[x.rindex('-', 0, -5)+1:x.rindex('-')]))
full_df['prompt'] =  full_df['model'].apply(lambda x: x[x.rindex('-', 0, -7)+1:x.rindex('-', 0, -5)])
full_df['dataset'] = full_df['dataset'].map({'unique': 'Deduplicate', 'duplicate': 'Original'})


full_df = full_df[full_df['shot'] < 6]

for group_index, (group_name, df) in enumerate( sorted(full_df.groupby('dataset'), key=lambda x: x[0], reverse=True)):
    print("\\hline")
    df = df.sort_values(by=['shot', 'prompt', 'model_order'], ascending=[True, True, True])
    shots = 0
    for loop_index, (index, row) in enumerate(df.iterrows()):
        model_url = row['model']
        # print(f"{len(df)} : {index}")
        is_middle_index = len(df) // 2 == loop_index
        if shots != int(row['shot']):
            print("\\cline{2-7}")
            shots = int(row['shot'])
        print(
            f"{row['dataset'] if is_middle_index else ''} & {convert_prompt(row['prompt'])} & {row['model_short']} & {row['shot']}    &   {bold_max('precision', row, df)} & {bold_max('recall', row, df)} & {bold_max('f1-score', row, df)}   & {row['support'] if is_middle_index else ''} \\\\")
print("\\hline")

\hline
 & No Keyword & t5-xl & 0    &   0.333 & 0.007 & 0.014   &  \\
 & No Keyword & t5-xxl & 0    &   0.103 & 0.657 & 0.179   &  \\
 & GPT 4 & t5-xl & 0    &   \textbf{ 1.0} & 0.036 & 0.07   &  \\
 & GPT 4 & t5-xxl & 0    &   0.198 & \textbf{ 0.9} & 0.323   &  \\
 & Jitterbug & t5-xl & 0    &   \textbf{ 1.0} & 0.467 & 0.637   &  \\
 & Jitterbug & t5-xxl & 0    &   0.283 & 0.832 & 0.422   &  \\
 & MAT & t5-xl & 0    &   0.986 & 0.518 & \textbf{ 0.7}   &  \\
 & MAT & t5-xxl & 0    &   0.328 & 0.803 & 0.466   &  \\
\cline{2-7}
 & No Keyword & t5-xl & 2    &   0.0 & 0.0 & 0.0   &  \\
 & No Keyword & t5-xxl & 2    &   0.192 & 0.504 & 0.278   &  \\
 & GPT 4 & t5-xl & 2    &   0.0 & 0.0 & 0.0   &  \\
 & GPT 4 & t5-xxl & 2    &   0.184 & 0.825 & 0.301   &  \\
Original & Jitterbug & t5-xl & 2    &   \textbf{ 1.0} & 0.109 & 0.197   & 137 \\
 & Jitterbug & t5-xxl & 2    &   0.273 & 0.818 & 0.409   &  \\
 & MAT & t5-xl & 2    &   \textbf{ 1.0} & 0.131 & 0.232   &  \\
 & MAT & t5-xxl & 2    &   0

In [ ]:
import pandas as pd
from pandas import DataFrame
def bold_max(column_name: str, row_df: DataFrame, df : DataFrame, decimal_digit: int = 1):
    value = row_df[column_name]
    if value == df[column_name].max():
        return "\\textbf{" + f"{value: .1f}" + "}"
    else:
        return value

MODEL_ORDER = ['gemini-2.0-flash','gemini-2.5-flash' , 'gpt-5-nano','gpt-5-mini' ,'gpt-5']
df = pd.read_csv('../cache/output/detect_output_metrics.csv')
full_df = df[(df['model'].str.contains('|'.join(['gpt', 'gemini']), regex= True)) & (df['metric'] == 'yes')].copy()
full_df['model_short'] = full_df['model'].apply(lambda x: x.split('/')[-1][:-7])
full_df['model_order'] = full_df['model_short'].apply(lambda x: MODEL_ORDER.index(x))
full_df['shot'] = full_df['model'].apply(lambda x: x[-6:-5])
full_df['dataset'] = full_df['dataset'].map({'unique': 'Deduplicate', 'duplicate': 'Original'})
full_df.head()

for group_index, (group_name, df) in enumerate( sorted(full_df.groupby('dataset'), key=lambda x: x[0], reverse=True)):
    # print("\\hline") if group_index > 0 else None
    print("\\hline")
    df = df.sort_values(by=['shot', 'model_order'], ascending=[True, True])
    shots = 0
    for loop_index, (index, row) in enumerate(df.iterrows()):
        model_url = row['model']
        # print(f"{len(df)} : {index}")
        is_middle_index = len(df) // 2 == loop_index
        if shots != int(row['shot']):
            print("\\cline{2-6}")
            shots = int(row['shot'])
        print(
            f"{row['dataset'] if is_middle_index else ''}  & {row['model_short']} & {row['shot']}    &   {bold_max('precision', row, df)} & {bold_max('recall', row, df)} & {bold_max('f1-score', row, df)}   & {row['support'] if is_middle_index else ''} \\\\")
print("\\hline")


In [ ]:
import pandas as pd
df = pd.read_csv('../data/duplicate_satd_comment.csv')
df = df[df['label'] == 'requirement']
todo_df = df[df['text'].str.lower().str.startswith('// todo')]
# df.head()
print(len(todo_df) /len(df))
